# NB03 — Coordinate Validation & QC

**Objective**: Validate all station coordinates from the wide table produced by NB02.

**Checks applied (in order)**:
1. Range check — lat ∈ [−90, 90], lon ∈ [−180, 180]
2. Null-island check — (lat=0, lon=0)
3. Country boundary check — station falls within its declared country polygon
4. Overseas territory flag — station outside mainland but in a known overseas territory
5. Near-duplicate detection — stations within 100 m of each other (GPS rounding artifacts)

**Policy**: No station is silently dropped. Every station receives a `QC_FLAG`.
Downstream notebooks filter on `QC_FLAG == 'VALID'` or `'OVERSEAS_TERRITORY'`.

**Inputs**:
- `data/intermediate/wosis_surface_wide.parquet` (from NB02)

**Outputs**:
- `data/intermediate/wosis_surface_qc.parquet` — wide table + `QC_FLAG` column
- `data/intermediate/audit/qc_flag_summary.csv` — count per flag per country
- `logs/nb03_coordinate_qc.log`

**STOP condition**:
- > 5% of stations flagged `INVALID_RANGE` or `NULL_ISLAND` → STOP (data integrity issue)
- `OUTSIDE_BORDER` > 15% for any single country → STOP and investigate

In [1]:
import glob as _glob
import logging
import sys
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from sklearn.neighbors import BallTree

# ── Paths ──────────────────────────────────────────────────────────────────
BASE      = Path('/home/agbelgaid/Documents/WORKSPACE/DataCollection/SoilHive')
INTER_DIR = BASE / 'data' / 'intermediate'
AUDIT_DIR = INTER_DIR / 'audit'
LOG_DIR   = BASE / 'logs'

# ── Logging ────────────────────────────────────────────────────────────────
log_path = LOG_DIR / 'nb03_coordinate_qc.log'
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler(log_path, mode='w'),
        logging.StreamHandler()
    ]
)
log = logging.getLogger('nb03')
log.info(f'NB03 started at {datetime.now().isoformat()}')

# ── QC flag values (explicit, exhaustive) ─────────────────────────────────
FLAG_VALID          = 'VALID'
FLAG_INVALID_RANGE  = 'INVALID_RANGE'       # lat/lon outside physical bounds
FLAG_NULL_ISLAND    = 'NULL_ISLAND'         # exactly (0.0, 0.0)
FLAG_NEAR_BORDER    = 'NEAR_BORDER'         # within 0.5° buffer — island/coastal artefact of lowres shapefile
FLAG_OUTSIDE_BORDER = 'OUTSIDE_BORDER'      # > 0.5° outside declared country polygon
FLAG_OVERSEAS       = 'OVERSEAS_TERRITORY'  # valid overseas territory of the country
FLAG_NEAR_DUPLICATE = 'NEAR_DUPLICATE'      # within 100 m of another station

# Flags considered usable for downstream pipeline
USABLE_FLAGS = {FLAG_VALID, FLAG_NEAR_BORDER, FLAG_OVERSEAS}

2026-03-19 21:00:48,135 [INFO] NB03 started at 2026-03-19T21:00:48.135296


In [2]:
# ── Load wide table from NB02 ──────────────────────────────────────────────
df = pd.read_parquet(INTER_DIR / 'wosis_surface_wide.parquet')
log.info(f'Loaded wosis_surface_wide.parquet: {len(df):,} stations')
print(f'Stations loaded: {len(df):,}')
print(f'Countries      : {df["iso3"].nunique()}')

# Initialise QC flag column — start all as VALID, then override
df['QC_FLAG'] = FLAG_VALID

2026-03-19 21:00:48,832 [INFO] Loaded wosis_surface_wide.parquet: 53,708 stations


Stations loaded: 53,708
Countries      : 23


In [3]:
# ── Check 1: Coordinate range ──────────────────────────────────────────────
invalid_range = (
    (df['lat'] < -90) | (df['lat'] > 90) |
    (df['lon'] < -180) | (df['lon'] > 180) |
    df['lat'].isna() | df['lon'].isna()
)
df.loc[invalid_range, 'QC_FLAG'] = FLAG_INVALID_RANGE

n_invalid = invalid_range.sum()
log.info(f'Check 1 — Invalid range: {n_invalid} stations')
print(f'Check 1 — INVALID_RANGE  : {n_invalid}')

2026-03-19 21:00:48,846 [INFO] Check 1 — Invalid range: 0 stations


Check 1 — INVALID_RANGE  : 0


In [4]:
# ── Check 2: Null island (0.0, 0.0) ───────────────────────────────────────
# Tolerance: within 0.001 degrees of origin (accounts for float rounding)
null_island = (
    (df['QC_FLAG'] == FLAG_VALID) &
    (df['lat'].abs() < 0.001) &
    (df['lon'].abs() < 0.001)
)
df.loc[null_island, 'QC_FLAG'] = FLAG_NULL_ISLAND

n_null = null_island.sum()
log.info(f'Check 2 — Null island: {n_null} stations')
print(f'Check 2 — NULL_ISLAND    : {n_null}')

2026-03-19 21:00:49,421 [INFO] Check 2 — Null island: 0 stations


Check 2 — NULL_ISLAND    : 0


In [5]:
# ── Load Natural Earth boundaries ──────────────────────────────────────────
def _find_naturalearth_shp() -> str:
    candidates = _glob.glob(
        f'{sys.prefix}/**/naturalearth_lowres/naturalearth_lowres.shp', recursive=True
    )
    if candidates:
        return candidates[0]
    candidates = _glob.glob(
        '/home/agbelgaid/anaconda3/**/naturalearth_lowres.shp', recursive=True
    )
    if candidates:
        return candidates[0]
    raise FileNotFoundError('naturalearth_lowres.shp not found')

NE_SHP = _find_naturalearth_shp()
world = gpd.read_file(NE_SHP)[['name', 'iso_a3', 'geometry']].copy()
world = world[world.geometry.notna()].reset_index(drop=True)

_ISO_FIXES = {
    'France': 'FRA', 'Norway': 'NOR',
    'N. Cyprus': 'CYP', 'Kosovo': 'XKX', 'Somaliland': 'SOM'
}
mask = world['iso_a3'] == '-99'
world.loc[mask, 'iso_a3'] = world.loc[mask, 'name'].map(_ISO_FIXES)

# Build a merged geometry per iso3 (union of all polygons for that country)
world_by_iso = world.dissolve(by='iso_a3')['geometry'].reset_index()
world_by_iso.columns = ['iso3', 'geometry']

log.info(f'Natural Earth loaded: {len(world_by_iso)} country polygons')
print(f'Natural Earth loaded: {len(world_by_iso)} country polygons')

2026-03-19 21:01:03,212 [INFO] Natural Earth loaded: 175 country polygons


Natural Earth loaded: 175 country polygons


In [6]:
# ── Check 3 & 4: Country boundary + overseas territory detection ───────────
#
# Strategy:
#   Pass 1 — Exact containment within the country polygon.
#   Pass 2 — Buffer containment (0.5°, ~55 km): catches stations on small islands
#             or within GPS/rounding distance of the border.
#             The Natural Earth lowres shapefile (110m resolution) does not capture
#             small islands (e.g. Greek Aegean islands, Canary Islands, etc.).
#             These are NOT flagged OUTSIDE_BORDER — they receive NEAR_BORDER.
#   Pass 3 — Overseas territory bounding boxes (France, Netherlands).
#   Remaining → OUTSIDE_BORDER.

BORDER_BUFFER_DEG = 0.5   # degrees — ~55 km at the equator

# Known overseas territory ISO3 pairs: (lat_min, lat_max, lon_min, lon_max)
OVERSEAS_BOXES = {
    'FRA': [
        (-6,  9, -55, -49),    # French Guiana
        (13, 19, -64, -59),    # Guadeloupe / Martinique
        (-22, -19, 53, 56),    # Réunion / Mayotte
        (-50, -48, 66, 68),    # Kerguelen
    ],
    'NLD': [
        (10, 19, -70, -60),    # Caribbean Netherlands (Bonaire, Saba, Sint Eustatius)
    ],
}

def is_overseas(lat: float, lon: float, iso3: str) -> bool:
    for (lat_min, lat_max, lon_min, lon_max) in OVERSEAS_BOXES.get(iso3, []):
        if lat_min <= lat <= lat_max and lon_min <= lon <= lon_max:
            return True
    return False

# Build buffered geometry per iso3 once (expensive, done outside the loop)
world_by_iso_buffered = world_by_iso.copy()
world_by_iso_buffered['geometry_buffered'] = world_by_iso_buffered['geometry'].buffer(BORDER_BUFFER_DEG)

# Lookup helpers
geom_exact   = world_by_iso.set_index('iso3')['geometry'].to_dict()
geom_buffered = world_by_iso_buffered.set_index('iso3')['geometry_buffered'].to_dict()

candidates = df[df['QC_FLAG'] == FLAG_VALID].copy()
log.info(f'Check 3+4 — Testing {len(candidates):,} stations against country boundaries '
         f'(buffer={BORDER_BUFFER_DEG}°)')

outside_border_idx = []
near_border_idx    = []   # within buffer but not exact
overseas_idx       = []

for iso3, group in candidates.groupby('iso3'):
    g_exact    = geom_exact.get(iso3)
    g_buffered = geom_buffered.get(iso3)

    if g_exact is None:
        log.warning(f'  {iso3}: no polygon in Natural Earth — {len(group)} stations → OUTSIDE_BORDER')
        outside_border_idx.extend(group.index.tolist())
        continue

    for idx, row in group.iterrows():
        pt = Point(row['lon'], row['lat'])

        if g_exact.contains(pt):
            continue  # stays VALID

        if is_overseas(row['lat'], row['lon'], iso3):
            overseas_idx.append(idx)

        elif g_buffered.contains(pt):
            # Within buffer zone — island or border station; geometry resolution artefact
            near_border_idx.append(idx)

        else:
            outside_border_idx.append(idx)

# Apply flags — do NOT touch VALID stations (already correct)
FLAG_NEAR_BORDER = 'NEAR_BORDER'   # within 0.5° buffer — likely island/coastal station
df.loc[near_border_idx,    'QC_FLAG'] = FLAG_NEAR_BORDER
df.loc[overseas_idx,       'QC_FLAG'] = FLAG_OVERSEAS
df.loc[outside_border_idx, 'QC_FLAG'] = FLAG_OUTSIDE_BORDER

log.info(
    f'Check 3+4 — OUTSIDE_BORDER: {len(outside_border_idx)}, '
    f'NEAR_BORDER: {len(near_border_idx)}, '
    f'OVERSEAS_TERRITORY: {len(overseas_idx)}'
)
print(f'Check 3 — OUTSIDE_BORDER    : {len(outside_border_idx)}')
print(f'Check 3 — NEAR_BORDER       : {len(near_border_idx)}  (island/coastal, buffer={BORDER_BUFFER_DEG}°)')
print(f'Check 4 — OVERSEAS_TERRITORY: {len(overseas_idx)}')

/tmp/ipykernel_1868302/2759592393.py:36: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  world_by_iso_buffered['geometry_buffered'] = world_by_iso_buffered['geometry'].buffer(BORDER_BUFFER_DEG)
2026-03-19 21:01:03,893 [INFO] Check 3+4 — Testing 53,708 stations against country boundaries (buffer=0.5°)
2026-03-19 21:01:07,081 [INFO] Check 3+4 — OUTSIDE_BORDER: 132, NEAR_BORDER: 1151, OVERSEAS_TERRITORY: 9


Check 3 — OUTSIDE_BORDER    : 132
Check 3 — NEAR_BORDER       : 1151  (island/coastal, buffer=0.5°)
Check 4 — OVERSEAS_TERRITORY: 9


In [7]:
# ── Check 5: Near-duplicate coordinates ───────────────────────────────────
# Detect stations within DISTANCE_THRESHOLD_M meters of another station
# in the same country. Likely GPS rounding artefacts in the source database.
#
# Only stations with USABLE flags are checked.
# The station with the lower index is kept; the other gets NEAR_DUPLICATE.

DISTANCE_THRESHOLD_M = 100   # meters
EARTH_RADIUS_M       = 6_371_000

near_dup_idx = []
to_check     = df[df['QC_FLAG'].isin(USABLE_FLAGS)].copy()

for iso3, group in to_check.groupby('iso3'):
    if len(group) < 2:
        continue

    coords_rad    = np.radians(group[['lat', 'lon']].values)
    tree          = BallTree(coords_rad, metric='haversine')
    threshold_rad = DISTANCE_THRESHOLD_M / EARTH_RADIUS_M
    indices       = tree.query_radius(coords_rad, r=threshold_rad)

    seen = set()
    for i, neighbours in enumerate(indices):
        orig_idx = group.index[i]
        if orig_idx in seen:
            continue
        for j in neighbours:
            if j != i:
                dup_idx = group.index[j]
                if dup_idx not in seen:
                    near_dup_idx.append(dup_idx)
                    seen.add(dup_idx)
        seen.add(orig_idx)

df.loc[near_dup_idx, 'QC_FLAG'] = FLAG_NEAR_DUPLICATE

log.info(f'Check 5 — NEAR_DUPLICATE: {len(near_dup_idx)} stations (threshold: {DISTANCE_THRESHOLD_M} m)')
print(f'Check 5 — NEAR_DUPLICATE    : {len(near_dup_idx)}  (within {DISTANCE_THRESHOLD_M} m)')

2026-03-19 21:01:07,871 [INFO] Check 5 — NEAR_DUPLICATE: 5141 stations (threshold: 100 m)


Check 5 — NEAR_DUPLICATE    : 5141  (within 100 m)


In [8]:
# ── STOP CONDITIONS ────────────────────────────────────────────────────────
flag_counts = df['QC_FLAG'].value_counts()
n_total     = len(df)

# HARD STOP: physical impossibility (invalid range or null island)
n_hard_errors   = flag_counts.get(FLAG_INVALID_RANGE, 0) + flag_counts.get(FLAG_NULL_ISLAND, 0)
pct_hard_errors = 100 * n_hard_errors / n_total

if pct_hard_errors > 5:
    raise AssertionError(
        f'STOP: {pct_hard_errors:.1f}% of stations have INVALID_RANGE or NULL_ISLAND '
        f'(threshold 5%). Data integrity issue — investigate before proceeding.'
    )

# WARNING only: OUTSIDE_BORDER > 15% for any country
# These are logged and kept in the dataset — they do NOT block the pipeline.
# Root causes: lowres shapefile precision, border stations, missing territories.
per_country = df.groupby('iso3')['QC_FLAG'].value_counts().unstack(fill_value=0)
per_country['total'] = per_country.sum(axis=1)
if FLAG_OUTSIDE_BORDER in per_country.columns:
    per_country['pct_outside'] = 100 * per_country[FLAG_OUTSIDE_BORDER] / per_country['total']
    bad_countries = per_country[per_country['pct_outside'] > 15]
    if len(bad_countries) > 0:
        log.warning(
            f'{len(bad_countries)} country/ies have > 15% OUTSIDE_BORDER stations '
            f'(WARNING — pipeline continues, downstream uses USABLE_FLAGS filter):\n'
            + bad_countries[['pct_outside', FLAG_OUTSIDE_BORDER, 'total']].to_string()
        )
        print(f'WARNING: {len(bad_countries)} country/ies with > 15% OUTSIDE_BORDER:')
        print(bad_countries[['pct_outside', FLAG_OUTSIDE_BORDER, 'total']].to_string())

log.info('STOP conditions checked — hard errors: 0, pipeline continues')
print(f'\nHard errors (INVALID_RANGE + NULL_ISLAND): {n_hard_errors} ({pct_hard_errors:.2f}%)')
print('Pipeline continues.')

2026-03-19 21:01:07,953 [INFO] STOP conditions checked — hard errors: 0, pipeline continues



Hard errors (INVALID_RANGE + NULL_ISLAND): 0 (0.00%)
Pipeline continues.


In [9]:
# ── Summary ────────────────────────────────────────────────────────────────
flag_counts = df['QC_FLAG'].value_counts().reset_index()
flag_counts.columns = ['QC_FLAG', 'count']
flag_counts['pct']  = (100 * flag_counts['count'] / n_total).round(2)

print('=== NB03 QC SUMMARY ===')
print(f'Total stations : {n_total:,}')
print()
print(flag_counts.to_string(index=False))

n_usable = df['QC_FLAG'].isin(USABLE_FLAGS).sum()
print(f'\nUsable for pipeline {set(USABLE_FLAGS)}: {n_usable:,} ({100*n_usable/n_total:.1f}%)')
log.info(f'Usable stations: {n_usable:,} / {n_total:,} ({100*n_usable/n_total:.1f}%)')

print('\nPer-country QC flag breakdown:')
breakdown = (
    df.groupby(['iso3', 'country', 'QC_FLAG'])
    .size()
    .reset_index(name='n')
    .pivot_table(index=['iso3', 'country'], columns='QC_FLAG', values='n', fill_value=0)
    .reset_index()
)
breakdown.columns.name = None
print(breakdown.to_string(index=False))

2026-03-19 21:01:08,921 [INFO] Usable stations: 48,435 / 53,708 (90.2%)


=== NB03 QC SUMMARY ===
Total stations : 53,708

           QC_FLAG  count   pct
             VALID  47368 88.20
    NEAR_DUPLICATE   5141  9.57
       NEAR_BORDER   1058  1.97
    OUTSIDE_BORDER    132  0.25
OVERSEAS_TERRITORY      9  0.02

Usable for pipeline {'NEAR_BORDER', 'OVERSEAS_TERRITORY', 'VALID'}: 48,435 (90.2%)

Per-country QC flag breakdown:
iso3              country  NEAR_BORDER  NEAR_DUPLICATE  OUTSIDE_BORDER  OVERSEAS_TERRITORY   VALID
 AGO               Angola         15.0           191.0             0.0                 0.0  1556.0
 ALB              Albania          3.0            16.0             0.0                 0.0    64.0
 AUS            Australia        623.0          2741.0            91.0                 0.0 33125.0
 BDI              Burundi          5.0             0.0             0.0                 0.0    23.0
 BEN                Benin         17.0             9.0             0.0                 0.0   699.0
 BFA         Burkina Faso         24.0           

In [10]:
# ── Detail: OUTSIDE_BORDER stations ───────────────────────────────────────
# Log all stations flagged OUTSIDE_BORDER so they can be reviewed.
# These are NOT dropped — they remain in the dataset with their flag.

outside = df[df['QC_FLAG'] == FLAG_OUTSIDE_BORDER][['station_id', 'iso3', 'country', 'lat', 'lon']]
if len(outside) > 0:
    print(f'\nOUTSIDE_BORDER stations ({len(outside)} total):')
    print(outside.to_string(index=False))
    log.info(f'OUTSIDE_BORDER detail:\n{outside.to_string()}')
else:
    print('\nNo OUTSIDE_BORDER stations.')
    log.info('No OUTSIDE_BORDER stations.')

overseas = df[df['QC_FLAG'] == FLAG_OVERSEAS][['station_id', 'iso3', 'country', 'lat', 'lon']]
if len(overseas) > 0:
    print(f'\nOVERSEAS_TERRITORY stations ({len(overseas)} total):')
    print(overseas.head(10).to_string(index=False))
    if len(overseas) > 10:
        print(f'  ... and {len(overseas)-10} more')
    log.info(f'OVERSEAS_TERRITORY: {len(overseas)} stations')

2026-03-19 21:01:26,738 [INFO] OUTSIDE_BORDER detail:
       station_id iso3    country        lat         lon
605    AUS_000605  AUS  Australia -40.215174  148.167992
606    AUS_000606  AUS  Australia -40.181779  148.201295
607    AUS_000607  AUS  Australia -40.165176  148.184597
608    AUS_000608  AUS  Australia -40.165174  148.084593
609    AUS_000609  AUS  Australia -40.148481  148.067997
610    AUS_000610  AUS  Australia -40.148478  148.134599
611    AUS_000611  AUS  Australia -40.148472  148.301297
612    AUS_000612  AUS  Australia -40.131778  148.184592
613    AUS_000613  AUS  Australia -40.115173  148.067992
614    AUS_000614  AUS  Australia -40.110970  148.228806
615    AUS_000615  AUS  Australia -40.098471  148.184601
616    AUS_000616  AUS  Australia -40.084443  143.914929
617    AUS_000617  AUS  Australia -40.081777  148.184593
618    AUS_000618  AUS  Australia -40.081776  148.001296
619    AUS_000619  AUS  Australia -40.067854  143.917730
620    AUS_000620  AUS  Australia 


OUTSIDE_BORDER stations (132 total):
station_id iso3   country        lat        lon
AUS_000605  AUS Australia -40.215174 148.167992
AUS_000606  AUS Australia -40.181779 148.201295
AUS_000607  AUS Australia -40.165176 148.184597
AUS_000608  AUS Australia -40.165174 148.084593
AUS_000609  AUS Australia -40.148481 148.067997
AUS_000610  AUS Australia -40.148478 148.134599
AUS_000611  AUS Australia -40.148472 148.301297
AUS_000612  AUS Australia -40.131778 148.184592
AUS_000613  AUS Australia -40.115173 148.067992
AUS_000614  AUS Australia -40.110970 148.228806
AUS_000615  AUS Australia -40.098471 148.184601
AUS_000616  AUS Australia -40.084443 143.914929
AUS_000617  AUS Australia -40.081777 148.184593
AUS_000618  AUS Australia -40.081776 148.001296
AUS_000619  AUS Australia -40.067854 143.917730
AUS_000620  AUS Australia -40.061202 144.051377
AUS_000621  AUS Australia -40.054157 143.919227
AUS_000622  AUS Australia -40.048097 144.032017
AUS_000623  AUS Australia -40.031771 148.134596
AU

In [11]:
# ── Save outputs ───────────────────────────────────────────────────────────
df.to_parquet(INTER_DIR / 'wosis_surface_qc.parquet', index=False)

# Save per-country flag summary
qc_summary = (
    df.groupby(['iso3', 'country', 'QC_FLAG'])
    .size()
    .reset_index(name='n_stations')
)
qc_summary['pct_of_country'] = (
    qc_summary['n_stations'] /
    qc_summary.groupby('iso3')['n_stations'].transform('sum') * 100
).round(2)
qc_summary.to_csv(AUDIT_DIR / 'qc_flag_summary.csv', index=False)

log.info(f'Saved wosis_surface_qc.parquet  — {len(df):,} rows, {df.columns.tolist()}')
log.info(f'Saved qc_flag_summary.csv       — {len(qc_summary)} rows')
log.info('NB03 completed successfully — proceed to NB04')

print('\nOutputs saved:')
for fname, path in [
    ('wosis_surface_qc.parquet', INTER_DIR / 'wosis_surface_qc.parquet'),
    ('qc_flag_summary.csv',      AUDIT_DIR  / 'qc_flag_summary.csv'),
]:
    print(f'  {fname}: {path.stat().st_size / 1e6:.2f} MB')

2026-03-19 20:48:29,775 [INFO] Saved wosis_surface_qc.parquet  — 53,708 rows, ['station_id', 'lat', 'lon', 'iso3', 'country', 'Al', 'BD', 'CEC', 'CF', 'Ca', 'CaCO3', 'Cu', 'EC', 'Fe', 'K', 'Mg', 'Mn', 'N', 'Na', 'P', 'TC', 'WR_gravimetric', 'WR_volumetric', 'clay', 'nematode', 'occ', 'pH', 'sand', 'silt', 'QC_FLAG']


2026-03-19 20:48:29,776 [INFO] Saved qc_flag_summary.csv       — 62 rows


2026-03-19 20:48:29,776 [INFO] NB03 completed successfully — proceed to NB04



Outputs saved:
  wosis_surface_qc.parquet: 2.17 MB
  qc_flag_summary.csv: 0.00 MB
